# Create the Northstar Semantic Model

Test checkpoint: creates a new Direct Lake on OneLake semantic model from the embedded Northstar template. Existing models are never overwritten.

Use a Fabric notebook session with Semantic Link available. The target workspace must have Fabric capacity. You need Contributor access to the workspace and access to the Warehouse data.

Edit the inputs in Cell 2 and select Run all. Leave the workspace name blank to use this notebook's workspace. No Lakehouse attachment or separate TMDL upload is needed.

## 1. Set Your Inputs

Choose the Warehouse and give the new semantic model a name that is not already in use. Leave `workspace_name` blank to use the workspace containing this notebook. The default model name is `sm_northstar_test`.


In [ ]:
workspace_name = ""
warehouse_name = "wh_northstar"
semantic_model_name = "sm_northstar_test"


## 2. Load the Libraries and Helpers

Load Semantic Link so the notebook can call Fabric and Power BI using your signed-in identity. The helper functions find workspace items, report API errors, and wait for operations to finish. Automatic request retries are disabled.


In [ ]:
import base64
import hashlib
import json
import re
import textwrap
import time
import zlib
from urllib.parse import urlparse
from uuid import UUID

try:
    import notebookutils
    import sempy.fabric as fabric
except ImportError as error:
    raise RuntimeError(
        "Run this notebook in Microsoft Fabric with Semantic Link available. "
        "Use the standard Fabric runtime rather than a local Python kernel."
    ) from error

fabric_client = fabric.FabricRestClient(retry_config={"total": 0})
power_bi_client = fabric.PowerBIRestClient(retry_config={"total": 0})
operation_timeout_seconds = 1800
model_id = None
model_created = False
refresh_completed = False


def require_success(response, action, allowed_statuses=(200,)):
    if response.status_code not in allowed_statuses:
        try:
            details = json.dumps(response.json(), ensure_ascii=True)
        except ValueError:
            details = response.text
        raise RuntimeError(
            f"{action} failed (HTTP {response.status_code}): {details[:1500]}. "
            "For access errors, check workspace permissions and Warehouse data access."
        )
    return response


def list_all(path):
    values = []
    while path:
        response = require_success(
            fabric_client.get(path, timeout=120), "List workspace items"
        )
        page = response.json()
        values.extend(page.get("value", []))
        continuation_token = page.get("continuationToken")
        if continuation_token:
            from urllib.parse import quote
            separator = "&" if "?" in path else "?"
            path = path.split("&continuationToken=")[0].split("?continuationToken=")[0]
            separator = "&" if "?" in path else "?"
            path = f"{path}{separator}continuationToken={quote(continuation_token, safe='')}"
        else:
            path = None
    return values


def unique_name(items, name, label):
    matches = [item for item in items if item["displayName"].casefold() == name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one {label} named {name!r}; found {len(matches)}. "
            "Check the input name and your access."
        )
    return matches[0]


def wait_for_operation(response, action):
    require_success(response, action, (200, 201, 202))
    if response.status_code != 202:
        return response.json()
    operation_id = response.headers.get("x-ms-operation-id") or response.headers.get("Operation-Id")
    if not operation_id:
        location = response.headers.get("Location", "")
        match = re.search(r"/operations/([0-9a-fA-F-]+)", location)
        if match:
            operation_id = match.group(1)
    if not operation_id:
        raise RuntimeError(
            f"{action} was accepted but no operation ID was returned. "
            "Inspect the workspace before rerunning; do not resubmit blindly."
        )
    operation_id = str(UUID(operation_id))
    print(f"{action} operation: {operation_id}")
    deadline = time.monotonic() + operation_timeout_seconds
    while time.monotonic() < deadline:
        state_response = require_success(
            fabric_client.get(f"/v1/operations/{operation_id}", timeout=120),
            f"Check {action} operation",
        )
        state = state_response.json()
        status = state.get("status")
        print(f"{action}: {status}")
        if status == "Succeeded":
            result = require_success(
                fabric_client.get(f"/v1/operations/{operation_id}/result", timeout=120),
                f"Get {action} result",
            )
            return result.json()
        if status in {"Failed", "Cancelled", "Canceled"}:
            raise RuntimeError(f"{action}: {json.dumps(state, ensure_ascii=True)}")
        delay = float(state_response.headers.get("Retry-After", 10))
        time.sleep(min(max(delay, 1), 30, max(0, deadline - time.monotonic())))
    raise TimeoutError(
        f"{action} is still pending after 30 minutes. Operation ID: {operation_id}. "
        "Inspect its status and the workspace before attempting another run."
    )


print("Semantic Link loaded. Existing models will not be updated or deleted.")


## 3. Load the Supplied Model Definition

This block holds the ready-made Northstar model and the functions used to unpack and prepare it. The embedded definition includes the tables, measures, relationships, and AI metadata. You do not need to edit the encoded model data or upload a separate TMDL file.


In [ ]:
template_bundle_zlib_base64 = "..."
template_bundle_sha256 = "a4d5f903dae65ad05731892bde899c5da3c361c81e2abb4abe95fb7230987233"

expected_tables = (
    "DimProperty", "DimGuest", "DimChannel", "DimExperience", "DimDate",
    "FactBooking", "FactStayNight", "FactPropertyInventory",
    "FactGuestSatisfaction", "FactPropertyBudget", "FactPropertyEvent",
)


def load_template():
    content = zlib.decompress(base64.b64decode(template_bundle_zlib_base64))
    if hashlib.sha256(content).hexdigest() != template_bundle_sha256:
        raise ValueError("The embedded model template failed its integrity check.")
    return json.loads(content)


def prepare_definition(template_files, workspace_id, warehouse_id):
    workspace_id = str(UUID(workspace_id))
    warehouse_id = str(UUID(warehouse_id))
    files = dict(template_files)
    required = {
        "definition.pbism", "definition/database.tmdl", "definition/model.tmdl",
        "definition/expressions.tmdl", "definition/relationships.tmdl",
        "definition/cultures/en-US.tmdl",
    } | {f"definition/tables/{table}.tmdl" for table in expected_tables}
    if set(files) != required:
        raise ValueError("Unexpected or missing embedded model definition parts.")
    database = files["definition/database.tmdl"]
    if not re.search(r"(?m)^[\t ]+compatibilityLevel: 1705[\t ]*$", database):
        raise ValueError("The model must retain compatibility level 1705.")
    tables_text = "\n".join(files[f"definition/tables/{table}.tmdl"] for table in expected_tables)
    modes = re.findall(r"(?m)^[\t ]+mode: (\S+)[\t ]*$", tables_text)
    if modes != ["directLake"] * len(expected_tables):
        raise ValueError("Every supplied partition must remain Direct Lake.")
    for setting in ("schemaName: dw", "expressionSource: DatabaseQuery"):
        if len(re.findall(r"(?m)^[\t ]+" + re.escape(setting) + r"[\t ]*$", tables_text)) != 11:
            raise ValueError(f"Unexpected partition source setting: {setting}")
    expressions = files["definition/expressions.tmdl"]
    pattern = (
        r'(AzureStorage\.DataLake\(")https://onelake\.dfs\.fabric\.microsoft\.com/'
        r'[0-9a-fA-F-]{36}/[0-9a-fA-F-]{36}("\))'
    )
    if len(re.findall(pattern, expressions)) != 1:
        raise ValueError("Expected one Direct Lake on OneLake source connection.")
    source_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{warehouse_id}"
    files["definition/expressions.tmdl"] = re.sub(
        pattern, lambda match: match.group(1) + source_url + match.group(2), expressions
    )
    parts = [
        {
            "path": path,
            "payload": base64.b64encode(content.encode("utf-8")).decode("ascii"),
            "payloadType": "InlineBase64",
        }
        for path, content in sorted(files.items())
    ]
    return {"format": "TMDL", "parts": parts}


## 4. Check the Workspace and Warehouse

Find the workspace and Warehouse from your inputs, check that the workspace has capacity, and confirm that the new model name is available. Then prepare the supplied model with the selected Warehouse's OneLake connection. This step does not create the model yet.


In [ ]:
for input_name, value in (
    ("workspace_name", workspace_name),
    ("warehouse_name", warehouse_name),
    ("semantic_model_name", semantic_model_name),
):
    if not isinstance(value, str):
        raise ValueError(f"{input_name} must be text.")
workspace_name = workspace_name.strip()
warehouse_name = warehouse_name.strip()
semantic_model_name = semantic_model_name.strip()
if not warehouse_name or not semantic_model_name:
    raise ValueError("Enter both a Warehouse name and a NEW semantic model name in Cell 2.")

if workspace_name:
    workspace = unique_name(list_all("/v1/workspaces"), workspace_name, "workspace")
    workspace_id = str(UUID(workspace["id"]))
else:
    workspace_id = str(UUID(notebookutils.runtime.context["currentWorkspaceId"]))

workspace = require_success(
    fabric_client.get(f"/v1/workspaces/{workspace_id}", timeout=120),
    "Check target workspace",
).json()
if not workspace.get("capacityId"):
    raise ValueError("The target workspace must be assigned to Fabric capacity.")

items = list_all(f"/v1/workspaces/{workspace_id}/items")
warehouse = unique_name(
    [item for item in items if item.get("type") == "Warehouse"],
    warehouse_name,
    "Warehouse",
)
warehouse_id = str(UUID(warehouse["id"]))
collisions = [
    item for item in items
    if item.get("type") == "SemanticModel"
    and item["displayName"].casefold() == semantic_model_name.casefold()
]
if collisions:
    raise ValueError(
        f"Semantic model {semantic_model_name!r} already exists. "
        "Choose a different name in Cell 2. Nothing was overwritten."
    )

template_bundle = load_template()
definition = prepare_definition(template_bundle["files"], workspace_id, warehouse_id)
print(f"Workspace: {workspace['displayName']} ({workspace_id})")
print(f"Warehouse: {warehouse_name} ({warehouse_id})")
print(f"NEW model: {semantic_model_name}")
print("Template: 11 Direct Lake tables, 106 columns, 59 measures, 22 relationships.")
print(f"Source export SHA-256: {template_bundle['source_sha256']}")
print("Source: Direct Lake on OneLake, dw schema. No SQL endpoint connection.")


## 5. Create the Semantic Model

Submit the prepared model definition to Fabric and wait for creation to finish. The notebook checks the name again before creating anything, then prints the new model ID and a workspace link. Existing semantic models are not updated or deleted.


In [ ]:
if model_created:
    raise RuntimeError(f"This session already created model {model_id}. Do not rerun creation.")
fresh_items = list_all(f"/v1/workspaces/{workspace_id}/items")
if any(
    item.get("type") == "SemanticModel"
    and item["displayName"].casefold() == semantic_model_name.casefold()
    for item in fresh_items
):
    raise ValueError(f"Model {semantic_model_name!r} already exists. Creation stopped.")

try:
    response = fabric_client.post(
        f"/v1/workspaces/{workspace_id}/semanticModels",
        json={
            "displayName": semantic_model_name,
            "description": "Northstar course checkpoint: Direct Lake on OneLake.",
            "definition": definition,
        },
        timeout=120,
    )
    created_model = wait_for_operation(response, "Create semantic model")
    model_id = str(UUID(created_model["id"]))
except Exception as error:
    raise RuntimeError(
        f"Creation did not finish successfully for {semantic_model_name!r}. "
        "The request was not automatically retried. Inspect the operation and workspace "
        "before rerunning; a new model may already exist. Nothing was updated or deleted."
    ) from error

model_created = True
print(f"Created model: {semantic_model_name} ({model_id})")
print(f"Workspace: https://app.fabric.microsoft.com/groups/{workspace_id}/list")
print("Next: initial Direct Lake refresh. Existing semantic models remain unchanged.")


## 6. Refresh the New Model

Start the initial Direct Lake refresh and wait for its result. This refresh frames the Warehouse data for the model rather than importing a copy. If the refresh fails, the new model remains in the workspace so you can inspect the error and its data access settings.


In [ ]:
if not model_created or not model_id:
    raise RuntimeError("Create the new model successfully before refreshing it.")

refresh_completed = False
refresh_path = f"/v1.0/myorg/groups/{workspace_id}/datasets/{model_id}/refreshes"
try:
    response = require_success(
        power_bi_client.post(
            refresh_path,
            json={"type": "full", "commitMode": "transactional", "retryCount": 0},
            timeout=120,
        ),
        "Start initial Direct Lake refresh",
        (202,),
    )
    location = response.headers.get("Location", "")
    if not location:
        raise RuntimeError("Refresh was accepted without a tracking Location header.")
    refresh_id = str(UUID(urlparse(location).path.rstrip("/").rsplit("/", 1)[-1]))
    print(f"Initial refresh request: {refresh_id}")
    deadline = time.monotonic() + operation_timeout_seconds
    previous_status = None
    while time.monotonic() < deadline:
        details_response = require_success(
            power_bi_client.get(f"{refresh_path}/{refresh_id}", timeout=120),
            "Check initial refresh",
            (200, 202),
        )
        details = details_response.json()
        status = details.get("status")
        if status != previous_status:
            print(f"Initial refresh: {status}")
            previous_status = status
        if status == "Completed":
            refresh_completed = True
            break
        if status in {"Failed", "Disabled", "Cancelled", "Canceled"}:
            raise RuntimeError(json.dumps(details, ensure_ascii=True)[:3000])
        time.sleep(min(10, max(0, deadline - time.monotonic())))
    else:
        raise TimeoutError(f"Refresh is still pending after 30 minutes: {refresh_id}")
except Exception as error:
    raise RuntimeError(
        f"Model {semantic_model_name!r} WAS CREATED ({model_id}), but its initial "
        "refresh did not finish successfully. It has NOT been deleted. "
        "Inspect the refresh, cloud connection/SSO, and Warehouse data access. "
        "No refresh request was automatically retried."
    ) from error

print("Initial Direct Lake refresh completed. No data was imported into this model.")


## 7. Verify the Saved Model

Read the new model definition back from Fabric and check its table, column, measure, and relationship counts. Also confirm that its partitions are still Direct Lake, its connection points to the selected Warehouse on OneLake, and its AI instructions are present.


In [ ]:
if not model_created or not refresh_completed:
    raise RuntimeError("Complete creation and initial refresh before validating the model.")

persisted_definition_validated = False
try:
    response = fabric_client.post(
        f"/v1/workspaces/{workspace_id}/semanticModels/{model_id}/getDefinition?format=TMDL",
        timeout=120,
    )
    saved = wait_for_operation(response, "Read saved model definition")
    saved_files = {
        part["path"]: base64.b64decode(part["payload"]).decode("utf-8-sig").replace("\r\n", "\n")
        for part in saved["definition"]["parts"]
        if part["path"].endswith(".tmdl")
    }
    saved_text = "\n".join(saved_files.values())
    inventory = {
        "tables": len(re.findall(r"(?m)^[\t ]*table ", saved_text)),
        "columns": len(re.findall(r"(?m)^[\t ]*column ", saved_text)),
        "measures": len(re.findall(r"(?m)^[\t ]*measure ", saved_text)),
        "relationships": len(re.findall(r"(?m)^[\t ]*relationship ", saved_text)),
    }
    if inventory != {"tables": 11, "columns": 106, "measures": 59, "relationships": 22}:
        raise ValueError(f"Saved model inventory does not match the template: {inventory}")
    modes = re.findall(r"(?m)^[\t ]+mode: (\S+)[\t ]*$", saved_text)
    if modes != ["directLake"] * 11:
        raise ValueError(f"Saved partitions are not all Direct Lake: {modes}")
    source_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{warehouse_id}"
    if f'AzureStorage.DataLake("{source_url}")' not in saved_files["definition/expressions.tmdl"]:
        raise ValueError("Saved model is not connected to the selected Warehouse on OneLake.")
    if "CustomInstructions" not in saved_text:
        raise ValueError("The saved model is missing the template's AI instructions.")
    persisted_definition_validated = True
except Exception as error:
    raise RuntimeError(
        f"Model {model_id} exists and was refreshed, but saved-definition validation failed. "
        "It has not been deleted. Inspect the error before continuing."
    ) from error

print("Saved definition verified: Direct Lake on OneLake, all objects and AI instructions present.")


## 8. Query the Model and View the Results

Run DAX queries to count rows in all 11 tables and evaluate Bookings, Customers Served, and Total Revenue. Display the results and a link to the new semantic model.


In [ ]:
if not persisted_definition_validated:
    raise RuntimeError("Validate the saved model definition before running the query checks.")

row_count_query = "EVALUATE UNION(\n" + ",\n".join(
    f'ROW("Table", "{table}", "Rows", COUNTROWS(\'{table}\'))'
    for table in expected_tables
) + "\n) ORDER BY [Table]"
measure_query = """
EVALUATE
ROW(
    "Bookings", 'FactBooking'[Bookings],
    "Customers Served", 'FactStayNight'[Customers Served],
    "Total Revenue", 'FactStayNight'[Total Revenue]
)
"""
try:
    row_counts = fabric.evaluate_dax(
        dataset=model_id, dax_string=row_count_query, workspace=workspace_id
    )
    measures = fabric.evaluate_dax(
        dataset=model_id, dax_string=measure_query, workspace=workspace_id
    )
    if len(row_counts) != 11 or len(measures) != 1:
        raise ValueError("The model queries returned an unexpected result shape.")
    display(row_counts)
    display(measures)
except Exception as error:
    raise RuntimeError(
        f"Model {model_id} was created, refreshed, and its definition verified, "
        "but a query check failed. The model has not been deleted. "
        "Check the underlying error, Warehouse data access, and semantic-model/XMLA access."
    ) from error

print(f"PASS: {semantic_model_name} created, refreshed, and queried successfully.")
print(f"Semantic model ID: {model_id}")
print(f"Model: https://app.powerbi.com/groups/{workspace_id}/datasets/{model_id}/details")
